In [1]:
import numpy as np
from moabb.datasets import EPFLP300
from moabb.paradigms import P300
from mne.decoding import Vectorizer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import cross_val_score, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

/Users/m/Desktop/cortext/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Channels that overlap with the Unicorn montage
channels = ["Fz", "C3", "Cz", "C4", "Pz", "PO3", "Oz", "PO4"]

dataset = EPFLP300()
paradigm = P300(
    resample=250,        # match the Unicorn sampling rate
    fmin=1, fmax=20,     # typical P300 band
    tmin=0, tmax=0.8,   # 0-800 ms after each flash
    channels=channels,
)

subject = 1
X, y, meta = paradigm.get_data(dataset=dataset, subjects=[subject])
print(X.shape, np.unique(y, return_counts=True))

ConnectionError: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))

In [ ]:
# Averaged waveform at Pz: the "peak" plot for your demo
pz = channels.index("Pz")
t = np.linspace(0, 0.8, X.shape[2])
import matplotlib.pyplot as plt
for label in np.unique(y):
    plt.plot(t, X[y == label][:, pz, :].mean(axis=0) * 1e6, label=label)
plt.xlabel("Time after flash (s)"); plt.ylabel("µV at Pz"); plt.legend()
plt.show()

In [ ]:
# Simple classifier: shrinkage LDA on the flattened epoch
clf = make_pipeline(
    Vectorizer(), StandardScaler(),
    LDA(solver="lsqr", shrinkage="auto"),
)

In [ ]:
# Keep each session together to avoid leakage
groups = meta["session"].values
scores = cross_val_score(
    clf, X, (y == "Target").astype(int),
    groups=groups, cv=GroupKFold(n_splits=4), scoring="roc_auc",
)
print("ROC AUC per fold:", scores.round(3), "mean:", scores.mean().round(3))